# Synapse Dedicated SQL Pool Demo (Executed from Spark via JDBC)
## Bronze / Silver / Gold + 20M Fact + SCD Type 2 (as-of OrderDate)

**Target Dedicated SQL pool**
- Server: `demosynapseazcomm.sql.azuresynapse.net`
- Database: `migrateMe`
- Auth: Managed Identity (AAD MSI)

**How it works**
- Notebook runs on a Spark pool.
- All DDL/DML is executed on the Dedicated SQL pool using JDBC.

**Run order**: top to bottom.


In [1]:
# Parameters
SERVER = 'demosynapseazcomm.sql.azuresynapse.net'
DATABASE = 'migrateMe'

# One-time initial load controls
INITIAL_LOAD_20M = True      # Set False after first run

# Daily batch delta controls
DELTA_ROWS_PER_RUN = 200000  # e.g., 200k/day for demo

print('SERVER:', SERVER)
print('DATABASE:', DATABASE)
print('INITIAL_LOAD_20M:', INITIAL_LOAD_20M)
print('DELTA_ROWS_PER_RUN:', DELTA_ROWS_PER_RUN)


StatementMeta(smallpool, 4, 2, Finished, Available, Finished)

SERVER: demosynapseazcomm.sql.azuresynapse.net
DATABASE: migrateMe
INITIAL_LOAD_20M: True
DELTA_ROWS_PER_RUN: 200000


In [8]:
SERVER = 'demosynapseazcomm.sql.azuresynapse.net'
DATABASE = 'migrateMe'

SQL_USER = 'demo_etl_user'
SQL_PASSWORD = 'Put-A-Strong-Password-Here!'  # store in Key Vault ideally

jdbc_url = (
    f"jdbc:sqlserver://{SERVER}:1433;"
    f"database={DATABASE};"
    "encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.sql.azuresynapse.net;"
    "loginTimeout=30;"
)

driver = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

def _get_conn():
    jvm = spark._sc._gateway.jvm
    jvm.java.lang.Class.forName(driver)
    props = jvm.java.util.Properties()
    props.setProperty("user", SQL_USER)
    props.setProperty("password", SQL_PASSWORD)
    return jvm.java.sql.DriverManager.getConnection(jdbc_url, props)

def exec_sql(sql: str):
    sql = sql.strip()
    if not sql:
        return
    conn = _get_conn()
    try:
        stmt = conn.createStatement()
        try:
            stmt.execute(sql)
        finally:
            stmt.close()
    finally:
        conn.close()

def exec_many(sql_list, title=None):
    if title:
        print(f"\n=== {title} ===")
    for i, s in enumerate(sql_list, 1):
        print(f"Running {i}/{len(sql_list)}")
        exec_sql(s)
    print("Done.")

StatementMeta(smallpool, 4, 9, Finished, Available, Finished)

## Connectivity test (Dedicated pool)
Confirms the notebook can execute a query via MSI.

In [9]:
exec_sql("SELECT DB_NAME() AS current_db, SUSER_SNAME() AS login_name, @@VERSION AS version")
print('Connectivity OK')


StatementMeta(smallpool, 4, 10, Finished, Available, Finished)

Connectivity OK


In [10]:
try:
    exec_sql("SELECT 1;")
except Exception as e:
    import traceback
    traceback.print_exc()
    # Also try to print the Java exception details if present
    je = getattr(e, "java_exception", None)
    if je is not None:
        print("\n--- java_exception.toString() ---")
        print(je.toString())
        print("\n--- java_exception.getMessage() ---")
        print(je.getMessage())

StatementMeta(smallpool, 4, 11, Finished, Available, Finished)

# ONE-TIME SETUP
Creates schemas, Numbers(20M), Bronze raw, Silver SCD2 dims, Silver fact, Gold aggregate.

In [23]:
setup_sql = []
# Numbers(20M) - Dedicated SQL pool compatible (no VALUES table constructor)
setup_sql += [
    # Drop old
    "IF OBJECT_ID('dbo.Numbers','U') IS NOT NULL DROP TABLE dbo.Numbers",
    "IF OBJECT_ID('dbo.NumbersSeed','U') IS NOT NULL DROP TABLE dbo.NumbersSeed",

    # Seed table (will grow beyond 20M)
    "CREATE TABLE dbo.NumbersSeed (n INT NOT NULL) WITH (DISTRIBUTION = ROUND_ROBIN, HEAP)",

    # Seed >= 1024 rows (sys views exist in dedicated pool)
    "INSERT INTO dbo.NumbersSeed(n) "
    "SELECT TOP (1024) ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) "
    "FROM sys.all_columns a CROSS JOIN sys.all_columns b",

    # Double until >= 20,000,000
    "WHILE (SELECT COUNT(1) FROM dbo.NumbersSeed) < 20000000 "
    "BEGIN "
    "  INSERT INTO dbo.NumbersSeed(n) "
    "  SELECT n + (SELECT COUNT(1) FROM dbo.NumbersSeed) "
    "  FROM dbo.NumbersSeed; "
    "END",

    # Final Numbers table exactly 20,000,000 rows
    "CREATE TABLE dbo.Numbers "
    "WITH (DISTRIBUTION = ROUND_ROBIN, HEAP) "
    "AS "
    "SELECT TOP (20000000) ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS n "
    "FROM dbo.NumbersSeed",

    # Cleanup seed
    "DROP TABLE dbo.NumbersSeed"
]

exec_batch(setup_sql, title='One-time setup (schemas, Numbers, bronze tables + dims)')

StatementMeta(smallpool, 4, 24, Finished, Available, Finished)


=== One-time setup (schemas, Numbers, bronze tables + dims) ===
Running 1/7...
Running 2/7...
Running 3/7...
Running 4/7...
Running 5/7...
Running 6/7...
Running 7/7...
Done.


In [25]:
setup_sql = []

# Schemas
setup_sql += [
    "IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'bronze') EXEC('CREATE SCHEMA bronze AUTHORIZATION dbo;')",
    "IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'silver') EXEC('CREATE SCHEMA silver AUTHORIZATION dbo;')",
    "IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'gold')   EXEC('CREATE SCHEMA gold AUTHORIZATION dbo;')"
]

# Bronze tables
setup_sql += [
    "IF OBJECT_ID('bronze.CustomerRaw','U') IS NOT NULL DROP TABLE bronze.CustomerRaw",
    "IF OBJECT_ID('bronze.ProductRaw','U')  IS NOT NULL DROP TABLE bronze.ProductRaw",
    "IF OBJECT_ID('bronze.StoreRaw','U')    IS NOT NULL DROP TABLE bronze.StoreRaw",
    "IF OBJECT_ID('bronze.SalesRaw','U')    IS NOT NULL DROP TABLE bronze.SalesRaw",

    "CREATE TABLE bronze.CustomerRaw (CustomerID INT NOT NULL, CustomerNK VARCHAR(20) NOT NULL, CustomerGroup VARCHAR(50) NOT NULL, Country VARCHAR(2) NOT NULL, Segment VARCHAR(20) NOT NULL, SourceSystem VARCHAR(20) NOT NULL, ExtractTS DATETIME2(0) NOT NULL) WITH (DISTRIBUTION = ROUND_ROBIN, HEAP)",
    "CREATE TABLE bronze.ProductRaw  (ProductID INT NOT NULL, ProductNK VARCHAR(20) NOT NULL, Category VARCHAR(30) NOT NULL, Subcategory VARCHAR(30) NOT NULL, ProductName VARCHAR(200) NOT NULL, ListPrice DECIMAL(10,2) NOT NULL, SourceSystem VARCHAR(20) NOT NULL, ExtractTS DATETIME2(0) NOT NULL) WITH (DISTRIBUTION = ROUND_ROBIN, HEAP)",
    "CREATE TABLE bronze.StoreRaw    (StoreID INT NOT NULL, StoreNK VARCHAR(20) NOT NULL, Channel VARCHAR(20) NOT NULL, Region VARCHAR(20) NOT NULL, SourceSystem VARCHAR(20) NOT NULL, ExtractTS DATETIME2(0) NOT NULL) WITH (DISTRIBUTION = ROUND_ROBIN, HEAP)",
    "CREATE TABLE bronze.SalesRaw    (SalesLineID BIGINT NOT NULL, OrderDate DATE NOT NULL, CustomerID INT NOT NULL, ProductID INT NOT NULL, StoreID INT NOT NULL, Qty SMALLINT NOT NULL, UnitPrice DECIMAL(10,2) NOT NULL, DiscountPct DECIMAL(6,4) NOT NULL, NetAmount DECIMAL(18,2) NOT NULL, SourceSystem VARCHAR(20) NOT NULL, ExtractTS DATETIME2(0) NOT NULL) WITH (DISTRIBUTION = ROUND_ROBIN, HEAP)"
]

# Populate bronze dims
setup_sql += [
    # CustomerRaw (no CHOOSE)
    "INSERT INTO bronze.CustomerRaw "
    "SELECT "
    "  n AS CustomerID, "
    "  CONCAT('C', RIGHT(CONCAT('000000', CAST(n AS VARCHAR(6))), 6)) AS CustomerNK, "
    "  CASE (n % 5) "
    "    WHEN 0 THEN 'Contoso' "
    "    WHEN 1 THEN 'Fabrikam' "
    "    WHEN 2 THEN 'Northwind' "
    "    WHEN 3 THEN 'AdventureWorks' "
    "    ELSE 'Litware' "
    "  END AS CustomerGroup, "
    "  CASE (n % 10) "
    "    WHEN 0 THEN 'DE' "
    "    WHEN 1 THEN 'FR' "
    "    WHEN 2 THEN 'NL' "
    "    WHEN 3 THEN 'UK' "
    "    WHEN 4 THEN 'ES' "
    "    WHEN 5 THEN 'IT' "
    "    WHEN 6 THEN 'SE' "
    "    WHEN 7 THEN 'NO' "
    "    WHEN 8 THEN 'PL' "
    "    ELSE 'US' "
    "  END AS Country, "
    "  CASE (n % 3) "
    "    WHEN 0 THEN 'Retail' "
    "    WHEN 1 THEN 'SMB' "
    "    ELSE 'Enterprise' "
    "  END AS Segment, "
    "  'ERP' AS SourceSystem, "
    "  SYSUTCDATETIME() AS ExtractTS "
    "FROM dbo.Numbers WHERE n <= 50000",

    # ProductRaw (no CHOOSE)
    "INSERT INTO bronze.ProductRaw "
    "SELECT "
    "  n AS ProductID, "
    "  CONCAT('P', RIGHT(CONCAT('0000000', CAST(n AS VARCHAR(7))), 7)) AS ProductNK, "
    "  CASE (n % 4) "
    "    WHEN 0 THEN 'Accessories' "
    "    WHEN 1 THEN 'Bikes' "
    "    WHEN 2 THEN 'Components' "
    "    ELSE 'Clothing' "
    "  END AS Category, "
    "  CASE (n % 5) "
    "    WHEN 0 THEN 'Road' "
    "    WHEN 1 THEN 'Mountain' "
    "    WHEN 2 THEN 'Touring' "
    "    WHEN 3 THEN 'Urban' "
    "    ELSE 'Kids' "
    "  END AS Subcategory, "
    "  CONCAT('Product-', n) AS ProductName, "
    "  CAST(5 + (ABS(CHECKSUM(n)) % 99500) / 100.0 AS DECIMAL(10,2)) AS ListPrice, "
    "  'ERP' AS SourceSystem, "
    "  SYSUTCDATETIME() AS ExtractTS "
    "FROM dbo.Numbers WHERE n <= 20000",

    # StoreRaw (no CHOOSE)
    "INSERT INTO bronze.StoreRaw "
    "SELECT "
    "  n AS StoreID, "
    "  CONCAT('S', RIGHT(CONCAT('00000', CAST(n AS VARCHAR(5))), 5)) AS StoreNK, "
    "  CASE (n % 3) "
    "    WHEN 0 THEN 'Online' "
    "    WHEN 1 THEN 'Retail' "
    "    ELSE 'Partner' "
    "  END AS Channel, "
    "  CASE (n % 5) "
    "    WHEN 0 THEN 'North' "
    "    WHEN 1 THEN 'South' "
    "    WHEN 2 THEN 'East' "
    "    WHEN 3 THEN 'West' "
    "    ELSE 'Central' "
    "  END AS Region, "
    "  'ERP' AS SourceSystem, "
    "  SYSUTCDATETIME() AS ExtractTS "
    "FROM dbo.Numbers WHERE n <= 400"
]

exec_batch(setup_sql, title='One-time setup (schemas, Numbers, bronze tables + dims)')


StatementMeta(smallpool, 4, 26, Finished, Available, Finished)


=== One-time setup (schemas, Numbers, bronze tables + dims) ===
Running 1/14...
Running 2/14...
Running 3/14...
Running 4/14...
Running 5/14...
Running 6/14...
Running 7/14...
Running 8/14...
Running 9/14...
Running 10/14...
Running 11/14...
Running 12/14...
Running 13/14...
Running 14/14...
Done.


## ONE-TIME: Load 20M into bronze.SalesRaw
Set `INITIAL_LOAD_20M = False` after first run.

In [26]:
if INITIAL_LOAD_20M:
    exec_batch([
        "INSERT INTO bronze.SalesRaw "
        "SELECT CAST(n AS BIGINT), DATEADD(DAY, (ABS(CHECKSUM(n)) % 1095), '2023-01-01'), "
        "1 + (ABS(CHECKSUM(n * 3)) % 50000), "
        "1 + (ABS(CHECKSUM(n * 5)) % 20000), "
        "1 + (ABS(CHECKSUM(n * 7)) % 400), "
        "CAST(1 + (ABS(CHECKSUM(n * 11)) % 5) AS SMALLINT), "
        "CAST(5 + (ABS(CHECKSUM(n * 13)) % 99500) / 100.0 AS DECIMAL(10,2)), "
        "CAST((ABS(CHECKSUM(n * 17)) % 2500) / 10000.0 AS DECIMAL(6,4)), "
        "CAST((1 + (ABS(CHECKSUM(n * 11)) % 5)) * (5 + (ABS(CHECKSUM(n * 13)) % 99500) / 100.0) * (1 - ((ABS(CHECKSUM(n * 17)) % 2500) / 10000.0)) AS DECIMAL(18,2)), "
        "'ERP', SYSUTCDATETIME() FROM dbo.Numbers"
    ], title='Initial 20M load into bronze.SalesRaw')
else:
    print('Skipped initial 20M load (INITIAL_LOAD_20M=False)')


StatementMeta(smallpool, 4, 27, Finished, Available, Finished)


=== Initial 20M load into bronze.SalesRaw ===
Running 1/1...
Done.


## ONE-TIME: Create Silver SCD2 dims, FactSales (CCI), watermark, initial fact load (as-of OrderDate)

In [27]:
silver_gold_sql = [
    "IF OBJECT_ID('silver.DimCustomer','U') IS NOT NULL DROP TABLE silver.DimCustomer",
    "IF OBJECT_ID('silver.DimProduct','U')  IS NOT NULL DROP TABLE silver.DimProduct",
    "IF OBJECT_ID('silver.DimStore','U')    IS NOT NULL DROP TABLE silver.DimStore",

    "CREATE TABLE silver.DimCustomer (CustomerSK INT IDENTITY(1,1) NOT NULL, CustomerNK VARCHAR(20) NOT NULL, CustomerGroup VARCHAR(50) NOT NULL, Country VARCHAR(2) NOT NULL, Segment VARCHAR(20) NOT NULL, EffectiveFrom DATETIME2(0) NOT NULL, EffectiveTo DATETIME2(0) NOT NULL, IsCurrent BIT NOT NULL) WITH (DISTRIBUTION = HASH(CustomerNK), HEAP)",
    "CREATE TABLE silver.DimProduct  (ProductSK INT IDENTITY(1,1) NOT NULL, ProductNK VARCHAR(20) NOT NULL, Category VARCHAR(30) NOT NULL, Subcategory VARCHAR(30) NOT NULL, ProductName VARCHAR(200) NOT NULL, ListPrice DECIMAL(10,2) NOT NULL, EffectiveFrom DATETIME2(0) NOT NULL, EffectiveTo DATETIME2(0) NOT NULL, IsCurrent BIT NOT NULL) WITH (DISTRIBUTION = HASH(ProductNK), HEAP)",
    "CREATE TABLE silver.DimStore    (StoreSK INT IDENTITY(1,1) NOT NULL, StoreNK VARCHAR(20) NOT NULL, Channel VARCHAR(20) NOT NULL, Region VARCHAR(20) NOT NULL, EffectiveFrom DATETIME2(0) NOT NULL, EffectiveTo DATETIME2(0) NOT NULL, IsCurrent BIT NOT NULL) WITH (DISTRIBUTION = HASH(StoreNK), HEAP)",

    "INSERT INTO silver.DimCustomer (CustomerNK, CustomerGroup, Country, Segment, EffectiveFrom, EffectiveTo, IsCurrent) SELECT CustomerNK, CustomerGroup, Country, Segment, SYSUTCDATETIME(), '9999-12-31', 1 FROM bronze.CustomerRaw",
    "INSERT INTO silver.DimProduct  (ProductNK, Category, Subcategory, ProductName, ListPrice, EffectiveFrom, EffectiveTo, IsCurrent) SELECT ProductNK, Category, Subcategory, ProductName, ListPrice, SYSUTCDATETIME(), '9999-12-31', 1 FROM bronze.ProductRaw",
    "INSERT INTO silver.DimStore    (StoreNK, Channel, Region, EffectiveFrom, EffectiveTo, IsCurrent) SELECT StoreNK, Channel, Region, SYSUTCDATETIME(), '9999-12-31', 1 FROM bronze.StoreRaw",

    "IF OBJECT_ID('silver.FactSales','U') IS NOT NULL DROP TABLE silver.FactSales",
    "CREATE TABLE silver.FactSales (SalesLineID BIGINT NOT NULL, OrderDate DATE NOT NULL, CustomerSK INT NOT NULL, ProductSK INT NOT NULL, StoreSK INT NOT NULL, Qty SMALLINT NOT NULL, UnitPrice DECIMAL(10,2) NOT NULL, DiscountPct DECIMAL(6,4) NOT NULL, NetAmount DECIMAL(18,2) NOT NULL, LoadTS DATETIME2(0) NOT NULL) WITH (DISTRIBUTION = HASH(CustomerSK), CLUSTERED COLUMNSTORE INDEX)",

    "IF OBJECT_ID('silver.ETL_Watermark','U') IS NOT NULL DROP TABLE silver.ETL_Watermark",
    "CREATE TABLE silver.ETL_Watermark (ProcessName VARCHAR(100) NOT NULL, LastSalesLineID BIGINT NOT NULL) WITH (DISTRIBUTION = REPLICATE, HEAP)",
    "INSERT INTO silver.ETL_Watermark(ProcessName, LastSalesLineID) VALUES ('FactSales', 0)",

    "DECLARE @Now DATETIME2(0) = SYSUTCDATETIME(); "
    "INSERT INTO silver.FactSales "
    "SELECT s.SalesLineID, s.OrderDate, c.CustomerSK, p.ProductSK, st.StoreSK, s.Qty, s.UnitPrice, s.DiscountPct, s.NetAmount, @Now "
    "FROM bronze.SalesRaw s "
    "JOIN silver.DimCustomer c ON c.CustomerNK = CONCAT('C', RIGHT(CONCAT('000000', CAST(s.CustomerID AS VARCHAR(6))), 6)) "
    " AND s.OrderDate >= CAST(c.EffectiveFrom AS DATE) AND s.OrderDate < CAST(c.EffectiveTo AS DATE) "
    "JOIN silver.DimProduct p ON p.ProductNK = CONCAT('P', RIGHT(CONCAT('0000000', CAST(s.ProductID AS VARCHAR(7))), 7)) "
    " AND s.OrderDate >= CAST(p.EffectiveFrom AS DATE) AND s.OrderDate < CAST(p.EffectiveTo AS DATE) "
    "JOIN silver.DimStore st ON st.StoreNK = CONCAT('S', RIGHT(CONCAT('00000', CAST(s.StoreID AS VARCHAR(5))), 5)) "
    " AND s.OrderDate >= CAST(st.EffectiveFrom AS DATE) AND s.OrderDate < CAST(st.EffectiveTo AS DATE); "
    "UPDATE silver.ETL_Watermark SET LastSalesLineID = (SELECT MAX(SalesLineID) FROM bronze.SalesRaw) WHERE ProcessName='FactSales'",

    "IF OBJECT_ID('gold.RptSalesMonthly','U') IS NOT NULL DROP TABLE gold.RptSalesMonthly",
    "CREATE TABLE gold.RptSalesMonthly (YearMonth CHAR(7) NOT NULL, Channel VARCHAR(20) NOT NULL, Region VARCHAR(20) NOT NULL, Country VARCHAR(2) NOT NULL, Segment VARCHAR(20) NOT NULL, Category VARCHAR(30) NOT NULL, Subcategory VARCHAR(30) NOT NULL, Units BIGINT NOT NULL, NetSales DECIMAL(18,2) NOT NULL, ActiveCustomers BIGINT NOT NULL) WITH (DISTRIBUTION = HASH(YearMonth), CLUSTERED COLUMNSTORE INDEX)",

    "INSERT INTO gold.RptSalesMonthly "
    "SELECT CONVERT(CHAR(7), f.OrderDate, 120), st.Channel, st.Region, c.Country, c.Segment, p.Category, p.Subcategory, "
    "SUM(CAST(f.Qty AS BIGINT)), SUM(f.NetAmount), COUNT(DISTINCT f.CustomerSK) "
    "FROM silver.FactSales f "
    "JOIN silver.DimStore st ON st.StoreSK = f.StoreSK "
    "JOIN silver.DimCustomer c ON c.CustomerSK = f.CustomerSK "
    "JOIN silver.DimProduct p ON p.ProductSK = f.ProductSK "
    "GROUP BY CONVERT(CHAR(7), f.OrderDate, 120), st.Channel, st.Region, c.Country, c.Segment, p.Category, p.Subcategory"
]

exec_batch(silver_gold_sql, title='Create Silver (SCD2 dims + fact) and Gold aggregate')


StatementMeta(smallpool, 4, 28, Finished, Available, Finished)


=== Create Silver (SCD2 dims + fact) and Gold aggregate ===
Running 1/18...
Running 2/18...
Running 3/18...
Running 4/18...
Running 5/18...
Running 6/18...
Running 7/18...
Running 8/18...
Running 9/18...
Running 10/18...
Running 11/18...
Running 12/18...
Running 13/18...
Running 14/18...
Running 15/18...
Running 16/18...
Running 17/18...
Running 18/18...
Done.


# DAILY BATCH (rerunnable)
Generates deltas, applies SCD2, loads incremental fact as-of OrderDate, refreshes gold.

In [28]:
daily_sql = [
    # Bronze sales delta
    f"DECLARE @DeltaRows BIGINT = {DELTA_ROWS_PER_RUN}; "
    "DECLARE @StartID BIGINT = (SELECT ISNULL(MAX(SalesLineID),0) FROM bronze.SalesRaw); "
    "INSERT INTO bronze.SalesRaw "
    "SELECT TOP (@DeltaRows) @StartID + n, "
    "DATEADD(DAY, -(ABS(CHECKSUM(n*31)) % 7), CAST(GETDATE() AS DATE)), "
    "1 + (ABS(CHECKSUM(n * 3)) % 50000), "
    "1 + (ABS(CHECKSUM(n * 5)) % 20000), "
    "1 + (ABS(CHECKSUM(n * 7)) % 400), "
    "CAST(1 + (ABS(CHECKSUM(n * 11)) % 5) AS SMALLINT), "
    "CAST(5 + (ABS(CHECKSUM(n * 13)) % 99500) / 100.0 AS DECIMAL(10,2)), "
    "CAST((ABS(CHECKSUM(n * 17)) % 2500) / 10000.0 AS DECIMAL(6,4)), "
    "CAST((1 + (ABS(CHECKSUM(n * 11)) % 5)) * (5 + (ABS(CHECKSUM(n * 13)) % 99500) / 100.0) * (1 - ((ABS(CHECKSUM(n * 17)) % 2500) / 10000.0)) AS DECIMAL(18,2)), "
    "'ERP', SYSUTCDATETIME() FROM dbo.Numbers;",

    # Bronze customer changes (~2%) to trigger SCD2
    "INSERT INTO bronze.CustomerRaw "
    "SELECT cr.CustomerID, cr.CustomerNK, cr.CustomerGroup, "
    "CASE WHEN (ABS(CHECKSUM(cr.CustomerID * 19)) % 2) = 0 THEN cr.Country ELSE 'US' END, "
    "CASE WHEN (ABS(CHECKSUM(cr.CustomerID * 23)) % 3) = 0 THEN 'Enterprise' ELSE cr.Segment END, "
    "cr.SourceSystem, SYSUTCDATETIME() "
    "FROM bronze.CustomerRaw cr "
    "WHERE cr.ExtractTS = (SELECT MAX(ExtractTS) FROM bronze.CustomerRaw) "
    "  AND (ABS(CHECKSUM(cr.CustomerID * 29)) % 50) = 0;",

    # Apply SCD2 Customer
    "DECLARE @Now DATETIME2(0) = SYSUTCDATETIME(); "
    ";WITH latest AS (SELECT CustomerNK, CustomerGroup, Country, Segment, ROW_NUMBER() OVER (PARTITION BY CustomerNK ORDER BY ExtractTS DESC) rn FROM bronze.CustomerRaw), "
    "src AS (SELECT CustomerNK, CustomerGroup, Country, Segment FROM latest WHERE rn = 1), "
    "chg AS (SELECT s.* FROM src s JOIN silver.DimCustomer d ON d.CustomerNK=s.CustomerNK AND d.IsCurrent=1 "
    "       WHERE d.CustomerGroup<>s.CustomerGroup OR d.Country<>s.Country OR d.Segment<>s.Segment) "
    "UPDATE d SET EffectiveTo=@Now, IsCurrent=0 FROM silver.DimCustomer d JOIN chg c ON c.CustomerNK=d.CustomerNK WHERE d.IsCurrent=1; "
    "INSERT INTO silver.DimCustomer (CustomerNK, CustomerGroup, Country, Segment, EffectiveFrom, EffectiveTo, IsCurrent) "
    "SELECT CustomerNK, CustomerGroup, Country, Segment, @Now, '9999-12-31', 1 FROM chg;",

    # Incremental fact load as-of OrderDate + watermark
    "DECLARE @Last BIGINT = (SELECT LastSalesLineID FROM silver.ETL_Watermark WHERE ProcessName='FactSales'); "
    "DECLARE @LoadTS DATETIME2(0) = SYSUTCDATETIME(); "
    "INSERT INTO silver.FactSales "
    "SELECT s.SalesLineID, s.OrderDate, c.CustomerSK, p.ProductSK, st.StoreSK, s.Qty, s.UnitPrice, s.DiscountPct, s.NetAmount, @LoadTS "
    "FROM bronze.SalesRaw s "
    "JOIN silver.DimCustomer c ON c.CustomerNK = CONCAT('C', RIGHT(CONCAT('000000', CAST(s.CustomerID AS VARCHAR(6))), 6)) "
    " AND s.OrderDate >= CAST(c.EffectiveFrom AS DATE) AND s.OrderDate < CAST(c.EffectiveTo AS DATE) "
    "JOIN silver.DimProduct p ON p.ProductNK = CONCAT('P', RIGHT(CONCAT('0000000', CAST(s.ProductID AS VARCHAR(7))), 7)) "
    " AND s.OrderDate >= CAST(p.EffectiveFrom AS DATE) AND s.OrderDate < CAST(p.EffectiveTo AS DATE) "
    "JOIN silver.DimStore st ON st.StoreNK = CONCAT('S', RIGHT(CONCAT('00000', CAST(s.StoreID AS VARCHAR(5))), 5)) "
    " AND s.OrderDate >= CAST(st.EffectiveFrom AS DATE) AND s.OrderDate < CAST(st.EffectiveTo AS DATE) "
    "WHERE s.SalesLineID > @Last; "
    "UPDATE silver.ETL_Watermark SET LastSalesLineID = (SELECT MAX(SalesLineID) FROM bronze.SalesRaw) WHERE ProcessName='FactSales';",

    # Gold refresh
    "TRUNCATE TABLE gold.RptSalesMonthly; "
    "INSERT INTO gold.RptSalesMonthly "
    "SELECT CONVERT(CHAR(7), f.OrderDate, 120), st.Channel, st.Region, c.Country, c.Segment, p.Category, p.Subcategory, "
    "SUM(CAST(f.Qty AS BIGINT)), SUM(f.NetAmount), COUNT(DISTINCT f.CustomerSK) "
    "FROM silver.FactSales f "
    "JOIN silver.DimStore st ON st.StoreSK=f.StoreSK "
    "JOIN silver.DimCustomer c ON c.CustomerSK=f.CustomerSK "
    "JOIN silver.DimProduct p ON p.ProductSK=f.ProductSK "
    "GROUP BY CONVERT(CHAR(7), f.OrderDate, 120), st.Channel, st.Region, c.Country, c.Segment, p.Category, p.Subcategory;"
]

exec_batch(daily_sql, title='DAILY BATCH: bronze delta -> SCD2 -> incremental fact -> gold refresh')


StatementMeta(smallpool, 4, 29, Finished, Available, Finished)


=== DAILY BATCH: bronze delta -> SCD2 -> incremental fact -> gold refresh ===
Running 1/5...
Running 2/5...
Running 3/5...
Running 4/5...
Running 5/5...
Done.


# Workload queries (run to simulate users)
You can run these repeatedly or call them from a scheduler.

In [29]:
workload = {
  'W1_gold_kpi': "SELECT TOP 6 * FROM gold.RptSalesMonthly ORDER BY YearMonth DESC",
  'W2_30day_channel_region': "SELECT st.Channel, st.Region, SUM(f.NetAmount) AS NetSales, SUM(CAST(f.Qty AS BIGINT)) AS Units FROM silver.FactSales f JOIN silver.DimStore st ON st.StoreSK=f.StoreSK WHERE f.OrderDate >= DATEADD(DAY,-30,CAST(GETDATE() AS DATE)) GROUP BY st.Channel, st.Region ORDER BY NetSales DESC",
  'W3_top_category_country_segment': "SELECT TOP 50 c.Country, c.Segment, p.Category, SUM(f.NetAmount) AS NetSales FROM silver.FactSales f JOIN silver.DimCustomer c ON c.CustomerSK=f.CustomerSK JOIN silver.DimProduct p ON p.ProductSK=f.ProductSK WHERE f.OrderDate >= DATEADD(MONTH,-6,CAST(GETDATE() AS DATE)) GROUP BY c.Country, c.Segment, p.Category ORDER BY NetSales DESC",
  'W4_90day_drilldown': "SELECT f.OrderDate, c.CustomerGroup, p.Category, p.Subcategory, st.Channel, st.Region, SUM(f.NetAmount) AS NetSales, COUNT_BIG(*) AS Lines FROM silver.FactSales f JOIN silver.DimCustomer c ON c.CustomerSK=f.CustomerSK JOIN silver.DimProduct p ON p.ProductSK=f.ProductSK JOIN silver.DimStore st ON st.StoreSK=f.StoreSK WHERE f.OrderDate >= DATEADD(DAY,-90,CAST(GETDATE() AS DATE)) GROUP BY f.OrderDate, c.CustomerGroup, p.Category, p.Subcategory, st.Channel, st.Region"
}

# Run one workload query as a quick test
exec_sql(workload['W1_gold_kpi'])
print('Ran W1_gold_kpi')


StatementMeta(smallpool, 4, 30, Finished, Available, Finished)

Ran W1_gold_kpi
